# 第57课 · 从一个神经元到一张网——MLP 从零搭建，41 个参数全由你亲手点亮

**目标**：实现 `Neuron → Layer → MLP`，每层支持前向传播（forward pass）和反向传播（`backward()`）。

🔗 **Aurora 连接**：这个 MLP 和 PyTorch `nn.Module` 完全同构，也是 Aurora ML 引擎（`src/aurora/ml/`，计划中模块）的原型；**L61** `L61_pytorch_nn.ipynb` 将用 PyTorch 复现相同的三层结构，届时你可以逐行对照两份代码，验证自己的实现和框架行为一致。

← **上一课**　[L56 · 反向传播手推](L56_backward_pass.ipynb)

> 上节课学习了 **反向传播手推**：链式法则逐层展开，梯度 = 局部梯度 × 上游梯度。  
> 本课将探讨 **MLP 从零搭建**。

## 本课剧情：从单个神经元到"会说话"的网络

有没有想过，为什么 GPT 里几十亿个参数都可以用同一行代码 `loss.backward()` 更新？

答案藏在它的**组装方式**里。神经网络不是一个大黑箱，而是三层嵌套：

```
Neuron  ← 最小单元：一个加权求和 + 激活函数
Layer   ← 中间层：nout 个 Neuron 并行，接收同一份输入
MLP     ← 完整网络：多个 Layer 首尾相连
```

**一个 Neuron 做的事**（nin=2，输出一个标量）：
```
output = relu( w[0]*x[0] + w[1]*x[1] + b )
       = relu( Value(0.7)*Value(1.0) + Value(-0.3)*Value(2.0) + Value(0.1) )
       = relu( Value(0.7 - 0.6 + 0.1) )
       = relu( Value(0.2) ) = Value(0.2)
```

每一步都是上节课实现的 `Value` 算子，计算图自动建立，`backward()` 自动传梯度。

**为什么这样设计**：

| 层级 | 角色 | 参数数 |
|---|---|---|
| Neuron(nin) | 1 个线性加权 + 激活 | nin+1 |
| Layer(nin, nout) | nout 个 Neuron | nout×(nin+1) |
| MLP(nin, [n1,n2,...]) | 多层串联 | 逐层累加 |

本节任务：实现三个 class，让 `MLP(3,[4,4,1])` 能前向、能 `backward()`、能数参数。

In [ ]:
import random

# ── Value 类（来自 L54–L56 的完整实现，这里直接粘贴以便本课独立运行）──
import math

class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __radd__(self, other): return self + other

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __rmul__(self, other): return self * other

    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return Value(other) + (-self)

    def __truediv__(self, other):
        # 必须用 other**-1（调用 __pow__），而非 Value(other.data**-1)。
        # Value(x**-1) 创建孤立节点：不连接到原来的 other，梯度无法回流。
        # other**-1 以 other 为子节点构建 1/other 节点，梯度链路完整。
        other = other if isinstance(other, Value) else Value(other)
        return self * other**-1

    def __pow__(self, exp):
        assert isinstance(exp, (int, float))
        out = Value(self.data**exp, (self,), f'**{exp}')
        def _backward():
            self.grad += exp * (self.data**(exp-1)) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), 'tanh')
        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Value(max(0, self.data), (self,), 'relu')
        def _backward():
            self.grad += (out.data > 0) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)
        build(self)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

# 快速冒烟测试
a = Value(2.0); b = Value(3.0)
c = (a * b + Value(1.0)).tanh()
c.backward()
assert abs(a.grad) > 0, "Value.backward() 未工作"
print("✅ Value 类加载成功")

# 除法梯度验证（节点连通性检查）
x = Value(6.0)
y = Value(2.0)
z = x / y          # z = 3.0
z.backward()
# d(x/y)/dx = 1/y = 0.5,  d(x/y)/dy = -x/y^2 = -1.5
assert abs(x.grad - 0.5) < 1e-9,   f"dz/dx 错误: {x.grad}（应为 0.5）"
assert abs(y.grad - (-1.5)) < 1e-9, f"dz/dy 错误: {y.grad}（应为 -1.5）"
print(f"✅ 除法梯度正确：dz/dx={x.grad:.4f}, dz/dy={y.grad:.4f}")

## 1. Neuron — 单个神经元

**实现要求**：

```python
class Neuron(nin, nonlin=True):
    self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
    self.b = Value(0.0)

    def __call__(x):  # x: list[Value or float]
        act = sum(wi * xi for wi, xi in zip(self.w, x)) + self.b
        return act.tanh() if self.nonlin else act

    def parameters() → list[Value]  # 返回 w + [b]
```

（micrograd 原版用 `relu`，本课统一用 `tanh` 作隐藏层激活——输出落在 (-1, 1)，便于观察；两者都是 L55 已实现的算子，可自行替换对比。）

**验收标准**：
- `Neuron(3)(x)` 返回单个 `Value`，且 `.data` 有限
- `len(Neuron(3).parameters()) == 4`（3 个 w + 1 个 b）
- `nonlin=False` 时直接返回线性组合，不过 tanh

In [ ]:
# 演示：手动搭一个 nin=2 的 Neuron（还没有 class，先把运算写出来）
random.seed(42)
w = [Value(random.uniform(-1, 1)) for _ in range(2)]
b  = Value(0.0)

x  = [Value(1.5), Value(-0.5)]

# 前向：dot + bias + tanh
act = sum((wi * xi for wi, xi in zip(w, x)), Value(0.0)) + b
out = act.tanh()

print("w  =", [round(wi.data, 4) for wi in w])
print("b  =", round(b.data, 4))
print("x  =", [xi.data for xi in x])
print("act=", round(act.data, 4))
print("out=", round(out.data, 4), "  (tanh 值在 (-1, 1) 之间)")

## 2. Layer — 并行的一组 Neuron

`Layer(nin, nout)` 包含 `nout` 个独立的 `Neuron(nin)`，它们接收**同一个输入向量 `x`**，各自独立计算，输出 `nout` 个标量组成的列表：

```
[neuron_0(x), neuron_1(x), ..., neuron_{nout-1}(x)]
```

`nout=1` 时输出列表长度为 1（最终回归层的常见设置）。每个 `Neuron` 的参数相互独立，梯度互不干扰。

In [ ]:
# 演示：手动搭 nout=3 的 Layer（nin=2 → 输出 3 个标量）
random.seed(0)
nin, nout = 2, 3
# 3 个 Neuron，每个有 2 个权重
neurons_demo = [
    [Value(random.uniform(-1, 1)) for _ in range(nin)]
    for _ in range(nout)
]
biases_demo = [Value(0.0) for _ in range(nout)]

x_demo = [Value(1.0), Value(-1.0)]

outputs_demo = []
for ws, b in zip(neurons_demo, biases_demo):
    act = sum((w * xi for w, xi in zip(ws, x_demo)), Value(0.0)) + b
    outputs_demo.append(act.tanh())

print(f"Layer(nin=2, nout=3) 输出 {len(outputs_demo)} 个 Value:")
for i, o in enumerate(outputs_demo):
    print(f"  out[{i}] = {o.data:.4f}")

## 3. MLP — 多层串联

`MLP(nin, nouts)` 把 `len(nouts)` 个 `Layer` 首尾相连：第 0 层接收原始输入（维度 `nin`），第 `k` 层接收第 `k-1` 层的输出。每层的 `nin` 恰好是上一层的 `nout`：

```
Layer(nin,       nouts[0])   → 输出 nouts[0] 维
Layer(nouts[0],  nouts[1])   → 输出 nouts[1] 维
...
Layer(nouts[-2], nouts[-1])  → 最终输出
```

最后一层通常关闭非线性（`nonlin=False`）。`parameters()` 遍历所有层的所有 `Neuron`，把全部 `w` 和 `b` 拍成一个平坦列表——训练时对这个列表里每个 `Value` 执行 `p.data -= lr * p.grad` 就完成一步梯度下降（gradient descent，GD）。

In [ ]:
# 演示：用嵌套列表模拟 MLP(3, [4, 4, 1]) 的层结构
# （先打印结构，后面才实现真正的 class）
layer_shapes = [(3, 4), (4, 4), (4, 1)]
total_params = sum(nin * nout + nout for nin, nout in layer_shapes)
print("MLP(3, [4, 4, 1]) 参数统计：")
for i, (n_in, n_out) in enumerate(layer_shapes):
    w_count = n_in * n_out
    b_count = n_out
    print(f"  Layer {i}: {n_in}×{n_out} 权重 + {n_out} 偏置 = {w_count + b_count}")
print(f"  总计：{total_params} 个可训练参数")

In [ ]:
class Neuron:
    def __init__(self, nin, nonlin=True):
        # ✏️ TODO: 初始化 self.w（nin 个 Value，均匀随机 [-1, 1)）和 self.b（Value(0.0)）
        # ✏️ TODO: 保存 self.nonlin
        raise NotImplementedError("TODO: 初始化 self.w（nin 个随机 Value）、self.b（Value(0.0)）并保存 self.nonlin")

    def __call__(self, x):
        # ✏️ TODO: 计算 act = sum(wi * xi for ...) + self.b
        # ✏️ TODO: 若 self.nonlin 为 True，返回 act.tanh()，否则返回 act
        raise NotImplementedError("TODO: 计算加权求和 act，若 nonlin 则返回 act.tanh()，否则返回 act")

    def parameters(self):
        # ✏️ TODO: 返回 self.w + [self.b]
        raise NotImplementedError("TODO: 返回 self.w + [self.b] 的参数列表")

    def __repr__(self):
        kind = "Tanh" if self.nonlin else "Linear"
        return f"{kind}Neuron({len(self.w)})"


In [ ]:
# ── Neuron 单元验证 ──
try:
    import random; random.seed(0)
    n = Neuron(3)
    assert len(n.w) == 3, f'Neuron(3) 应有 3 个权重，实际 {len(n.w)}'
    assert isinstance(n.b, Value), 'b 应为 Value'
    out = n([Value(1.0), Value(0.5), Value(-1.0)])
    assert isinstance(out, Value), '__call__ 应返回 Value'
    params = n.parameters()
    assert len(params) == 4, f'Neuron(3) 应有 4 个参数，实际 {len(params)}'
    print('✅ Neuron 通过：w×3 + b，__call__ 返回 Value，parameters() 长度=4')
except NotImplementedError:
    print('⬜ Neuron 未实现')
except NameError:
    print('⬜ 请先运行定义 Value 的 cell（来自 L55/L56）')


In [ ]:
class Layer:
    def __init__(self, nin, nout, **kwargs):
        # ✏️ TODO: self.neurons = nout 个 Neuron(nin, **kwargs)
        raise NotImplementedError("TODO: 创建 self.neurons，包含 nout 个 Neuron(nin, **kwargs)")

    def __call__(self, x):
        # ✏️ TODO: outs = [n(x) for n in self.neurons]
        # ✏️ TODO: 若 len(outs)==1 返回 outs[0]，否则返回 outs
        raise NotImplementedError("TODO: 前向传播：列表推导得 outs，len==1 时返回标量否则返回列表")

    def parameters(self):
        # ✏️ TODO: 返回所有 neuron 的 parameters() 拍平后的列表
        raise NotImplementedError("TODO: 展平所有 neuron.parameters() 并返回单一列表")

    def __repr__(self):
        return f"Layer([{', '.join(str(n) for n in self.neurons)}])"


In [ ]:
# ── Layer 单元验证 ──
try:
    import random; random.seed(1)
    layer = Layer(2, 3)
    assert len(layer.neurons) == 3, f'Layer(2,3) 应有 3 个 Neuron，实际 {len(layer.neurons)}'
    outs = layer([Value(0.5), Value(-0.5)])
    assert isinstance(outs, list) and len(outs) == 3, 'Layer(2,3) 输出应为长度 3 的列表'
    assert len(layer.parameters()) == 9, f'Layer(2,3) 应有 9 个参数，实际 {len(layer.parameters())}'
    print('✅ Layer 通过：3 neurons，输出列表长=3，parameters() 长度=9')
    # 单输出层应返回 Value 而非列表
    single = Layer(2, 1)([Value(0.5), Value(-0.5)])
    assert isinstance(single, Value), 'Layer(2,1) 应返回单个 Value'
    print('✅ Layer(2,1) 单输出时返回 Value（非列表）')
except NotImplementedError:
    print('⬜ Layer 未实现')
except NameError:
    print('⬜ 请先实现 Neuron')


In [ ]:
class MLP:
    def __init__(self, nin, nouts):
        # ✏️ TODO: sz = [nin] + nouts
        # ✏️ TODO: self.layers = Layer(sz[i], sz[i+1], nonlin=...) 列表
        #          最后一层 nonlin=False，其余 True
        raise NotImplementedError("TODO: 构建 sz 列表，创建 self.layers（最后一层 nonlin=False，其余 True）")

    def __call__(self, x):
        # ✏️ TODO: 依次把 x 喂入每一层（x = layer(x)）
        # ✏️ TODO: 统一返回列表（若结果不是 list 则 [x]）
        raise NotImplementedError("TODO: 串联各层前向传播，最终结果统一包装为列表返回")

    def parameters(self):
        # ✏️ TODO: 返回所有层所有参数的平坦列表
        raise NotImplementedError("TODO: 展平所有 layer.parameters() 并返回单一列表")

    def __repr__(self):
        return f"MLP([{', '.join(str(l) for l in self.layers)}])"


In [ ]:
# ── MLP 综合验证（未实现时打印提示而非崩溃） ──
try:
    random.seed(0)

    # 基础结构检查
    m = MLP(2, [3, 1])
    out = m([Value(0.5), Value(-0.5)])
    assert isinstance(out, list) and len(out) == 1, "MLP 输出应为长度 1 的列表"
    assert isinstance(out[0], Value), "MLP 输出元素应为 Value"
    print("✅ MLP(2,[3,1]) 前向传播通过，out[0].data =", round(out[0].data, 4))

    # 参数数量检查
    m2 = MLP(3, [4, 4, 1])
    params = m2.parameters()
    assert len(params) == 41, f"参数数量应为 41，实际 {len(params)}"
    print("✅ MLP(3,[4,4,1]) 参数数量正确：", len(params))

    # 反向传播检查
    out2 = m2([Value(1.0), Value(2.0), Value(3.0)])
    loss = out2[0]
    loss.backward()
    grads = [p.grad for p in params]
    assert any(g != 0.0 for g in grads), "backward() 后参数梯度应不全为 0"
    print("✅ backward() 梯度传播正常，首个参数梯度 =", round(params[0].grad, 6))
except (NotImplementedError, TypeError):
    print('⬜ TODO：请先完成 Neuron / Layer / MLP 三个 class，再运行本格验证。')

## 参数实验：验证 MLP(3,[4,4,1]) 的参数总数

手动推算三层的参数数量，再用 `len(m.parameters())` 核对：

| 层 | nin | nout | 权重数 `nin × nout` | 偏置（bias，b）数 `nout` | 小计 |
|----|-----|------|---------------------|---------------|------|
| 0  |  3  |  4   | 12                  | 4             | **16** |
| 1  |  4  |  4   | 16                  | 4             | **20** |
| 2  |  4  |  1   |  4                  | 1             | **5**  |
| 合计 | — | — | — | — | **41** |

预期现象：`len(m.parameters()) == 41`，且 `Neuron.__repr__` 最后一层显示 `LinearNeuron`（无非线性）。

In [ ]:
# ── 参数总数核对（未实现时打印提示而非崩溃） ──
try:
    random.seed(99)
    m3 = MLP(3, [4, 4, 1])
    print("网络结构：", m3)
    print()

    for i, layer in enumerate(m3.layers):
        lp = layer.parameters()
        print(f"Layer {i}: {len(lp):3d} 个参数  |  neurons: {layer.neurons}")

    total = len(m3.parameters())
    print(f"\n总参数：{total}  （预期 41）")

    # 检查最后一层是否关闭了非线性
    assert not m3.layers[-1].neurons[0].nonlin, "最后一层 Neuron 应 nonlin=False（线性输出）"
    print("✅ 最后一层为线性（nonlin=False）")
except (NotImplementedError, TypeError):
    print('⬜ TODO：请先完成 Neuron / Layer / MLP 三个 class，再运行本格核对参数数。')

## 本课收束

本节实现了 `Neuron`（单个加权求和+激活）、`Layer`（并行 `nout` 个 Neuron）、`MLP`（多层串联），`parameters()` 返回全部 41 个可训练 `Value` 的平坦列表，让梯度下降一个个更新它们。这套结构和 PyTorch `nn.Module` 完全同构，也是 Aurora ML 引擎（`src/aurora/ml/`，计划中模块）里网络主体的原型。下一节 `L58_training_loop.ipynb` 将用这个 MLP 拟合 `make_moons` 双月牙数据集（sklearn 缺失时用 NumPy 从零生成），以 hinge loss 为目标，串起「前向 → 算 loss → backward() → 更新参数 → 清零梯度」的完整训练循环，验收标准是分类准确率 > 85%。

## ✏️ 白板挑战：MLP 结构手推（目标 10 分钟）

盖上屏幕，纸上完成：

**问 1**：`MLP(2, [3, 1])` 共有多少个参数？（列出每层的 weight + bias 数）  
（Layer1: 3×(2+1)=9，Layer2: 1×(3+1)=4，共 13 个）

**问 2**：`Layer(4, 5)` 的 `forward(x)` 输出是什么形状？输入 x 是什么形状？  
（输入：4个标量的列表，输出：5个 Value 的列表）

**问 3**：为什么最后一层 `MLP` 通常 `nonlin=False`？  
（最后一层是回归输出或 logit，不需要 relu 截断负值）

**问 4**：`Neuron.parameters()` 返回什么？为什么需要这个方法？  
（返回 w + [b]，训练时需要遍历所有参数做 `p.grad = 0.0` 清零）

**问 5**：若 `MLP(3, [4,4,1])` 输入 `[1.0, 2.0, 3.0]`，前向传播后有多少个 `Value` 节点（大约）？  
（每个 Neuron 约 nin+1 次加法+1次激活，第1层4个Neuron≈20节点，第2层4个≈20，第3层1个≈5，合计约45个节点）

推导完成后运行下方格验证。

In [ ]:
# ✏️ 对答案格
import random

# 问1：MLP(2,[3,1]) 参数数 = 13
try:
    random.seed(0)
    m_check = MLP(2, [3, 1])
    params = m_check.parameters()
    assert len(params) == 13, f"参数数={len(params)}，期望13"
    print(f"Q1 ✅  MLP(2,[3,1]) 参数数={len(params)} (Layer1:9 + Layer2:4 = 13)")
except NotImplementedError:
    print(f"Q1 ✅  MLP(2,[3,1])参数数: Layer1=3×(2+1)=9, Layer2=1×(3+1)=4, 共13（待实现后验证）")

# 问2：Layer(4,5)输出5个Value
try:
    random.seed(1)
    layer_check = Layer(4, 5)
    x_in = [Value(float(i)) for i in range(4)]
    out_check = layer_check(x_in)
    assert len(out_check) == 5, f"输出长度={len(out_check)}"
    print(f"Q2 ✅  Layer(4,5): 输入4个Value → 输出{len(out_check)}个Value")
except NotImplementedError:
    print(f"Q2 ✅  Layer(4,5): 输入列表长4，输出列表长5（待实现后验证）")

# 问3：最后一层nonlin=False（概念验证）
try:
    random.seed(2)
    mlp_check = MLP(2, [3, 1])
    last_layer = mlp_check.layers[-1]
    assert all(not n.nonlin for n in last_layer.neurons), "最后层应nonlin=False"
    print(f"Q3 ✅  MLP最后层所有Neuron.nonlin=False（回归输出不截断）")
except NotImplementedError:
    print(f"Q3 ✅  最后层nonlin=False：回归logit不需relu截断（待实现后验证）")

# 问4：Neuron.parameters()长度
try:
    random.seed(3)
    n_check = Neuron(5)
    assert len(n_check.parameters()) == 6, f"参数数={len(n_check.parameters())}"
    print(f"Q4 ✅  Neuron(5).parameters()={len(n_check.parameters())}个（5个w + 1个b）")
except NotImplementedError:
    print(f"Q4 ✅  Neuron(nin).parameters() = [w0,...,w_(nin-1), b]，共nin+1个（待实现后验证）")

# 问5：参数总数 MLP(3,[4,4,1])
try:
    random.seed(4)
    m3_check = MLP(3, [4, 4, 1])
    n_params = len(m3_check.parameters())
    # L1: 4×(3+1)=16, L2: 4×(4+1)=20, L3: 1×(4+1)=5 → 41
    assert n_params == 41, f"参数数={n_params}"
    print(f"Q5 ✅  MLP(3,[4,4,1]) 参数数={n_params} (L1:16 + L2:20 + L3:5 = 41)")
except NotImplementedError:
    print(f"Q5 ✅  MLP(3,[4,4,1]): L1=4×4=16, L2=4×5=20, L3=1×5=5, 共41（待实现后验证）")
print("\n🎉 MLP 结构白板挑战通过！")

In [ ]:
# ✏️ 本课自评
l57_review = {
    "neuron_class_impl":       None,  # Neuron.__init__/__call__/parameters 正确实现？True/False
    "layer_class_impl":        None,  # Layer.__init__/__call__/parameters 正确实现？True/False
    "mlp_class_impl":          None,  # MLP.__init__/__call__/parameters 正确实现？True/False
    "param_count_understood":  None,  # 能手算 MLP 任意结构的参数总数？True/False
    "whiteboard_passed":       None,  # 白板挑战5道手推全部完成？True/False
}

unfilled = [k for k, v in l57_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l57_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L57 全部通关！进入 L58：训练循环')

---

→ **下一课**　[L58 · 训练循环](L58_training_loop.ipynb)

> 下节课将学习 **训练循环**：loss 计算、optimizer.step、收敛曲线，拟合 make_moons 数据集。